# Chapter 3 — Governments Make Their Choices

The COVID-19 pandemic forced governments to make difficult decisions about public health, economic activity, and social restrictions. Countries introduced different combinations of measures, including school and workplace closures, travel restrictions, and income support.

This chapter examines how government responses changed between 2020 and 2022 across the 11 countries in our study. Using data from the Oxford COVID-19 Government Response Tracker (OxCGRT), we compare the strictness of policy responses and the level of reported economic support over time.

The goal is to explore how countries chose to respond to the same global crisis, while recognizing that policy indices describe government measures and do not, by themselves, establish how effective those measures were.

In [47]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"

policy = pd.read_csv(DATA_DIR / "oxcgrt_policy_2020_2022.csv")
gdp = pd.read_csv(DATA_DIR / "world_bank_gdp_growth.csv")

print("GDP data shape:", gdp.shape)
print("GDP columns:", gdp.columns.tolist())
print("Policy data shape:", policy.shape)
print("\nColumns:")
print(policy.columns.tolist())

display(policy.head())

GDP data shape: (143, 4)
GDP columns: ['ISO3', 'country', 'year', 'gdp_growth_pct']
Policy data shape: (12056, 15)

Columns:
['CountryName', 'CountryCode', 'Date', 'C1M_School closing', 'C2M_Workplace closing', 'C3M_Cancel public events', 'C4M_Restrictions on gatherings', 'C5M_Close public transport', 'C6M_Stay at home requirements', 'C7M_Restrictions on internal movement', 'C8EV_International travel controls', 'E1_Income support', 'E2_Debt/contract relief', 'StringencyIndex_Average', 'EconomicSupportIndex']


,CountryName,CountryCode,Date,C1M_School closing,C2M_Workplace closing,C3M_Cancel public events,C4M_Restrictions on gatherings,C5M_Close public transport,C6M_Stay at home requirements,C7M_Restrictions on internal movement,C8EV_International travel controls,E1_Income support,E2_Debt/contract relief,StringencyIndex_Average,EconomicSupportIndex
0,Brazil,BRA,2020-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Brazil,BRA,2020-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Brazil,BRA,2020-01-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Brazil,BRA,2020-01-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Brazil,BRA,2020-01-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Data Preparation and Quality Checks

Before comparing government responses, we check the dataset's date range, country coverage, duplicate records, and missing values. We also convert the date column into a datetime format and ensure that policy indicators are numeric.

These checks help identify gaps and prevent missing observations from being mistaken for an absence of government action.

In [48]:
# Convert dates and policy indicators to the correct data types
policy["Date"] = pd.to_datetime(policy["Date"], errors="coerce")

policy_columns = [
    "C1M_School closing",
    "C2M_Workplace closing",
    "C3M_Cancel public events",
    "C4M_Restrictions on gatherings",
    "C5M_Close public transport",
    "C6M_Stay at home requirements",
    "C7M_Restrictions on internal movement",
    "C8EV_International travel controls",
    "E1_Income support",
    "E2_Debt/contract relief",
    "StringencyIndex_Average",
    "EconomicSupportIndex",
]

for col in policy_columns:
    policy[col] = pd.to_numeric(policy[col], errors="coerce")

# Check date range and country coverage
print("Date range:", policy["Date"].min(), "to", policy["Date"].max())
print("Number of countries:", policy["CountryCode"].nunique())
print("Countries:", sorted(policy["CountryName"].dropna().unique()))

# Check duplicate country-date records
print(
    "Duplicate country-date records:",
    policy.duplicated(subset=["CountryCode", "Date"]).sum()
)

# Check missing values
missing_summary = pd.DataFrame({
    "Missing count": policy.isna().sum(),
    "Missing percentage": (policy.isna().mean() * 100).round(2)
})

display(missing_summary)

Date range: 2020-01-01 00:00:00 to 2022-12-31 00:00:00
Number of countries: 11
Countries: ['Brazil', 'China', 'Germany', 'India', 'Saudi Arabia', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Thailand', 'United States']
Duplicate country-date records: 0


,Missing count,Missing percentage
CountryName,0,0.0
CountryCode,0,0.0
Date,0,0.0
C1M_School closing,0,0.0
C2M_Workplace closing,0,0.0
C3M_Cancel public events,0,0.0
C4M_Restrictions on gatherings,0,0.0
C5M_Close public transport,0,0.0
C6M_Stay at home requirements,0,0.0
C7M_Restrictions on internal movement,0,0.0


## 1. Government Policy Stringency Over Time (2020–2022)

Governments introduced different public-health measures during the COVID-19 pandemic, including school closures, workplace restrictions, limits on gatherings, and travel controls. The timing and strictness of these measures varied across countries.

The OxCGRT Stringency Index summarizes the strictness of selected government policies on a scale from 0 to 100. Higher values indicate stricter policies, but the index does not measure policy effectiveness or public compliance.

This section compares monthly average policy stringency across the 11 countries from 2020 to 2022 to identify differences in the timing and intensity of government responses.

In [49]:
# Calculate monthly average government policy stringency

policy["Month"] = policy["Date"].dt.to_period("M").dt.to_timestamp()

stringency_monthly = (
    policy.groupby(["CountryName", "Month"], as_index=False)
    ["StringencyIndex_Average"]
    .mean()
)

print("Number of countries:", stringency_monthly["CountryName"].nunique())
print("Number of monthly observations:", len(stringency_monthly))

display(stringency_monthly.head(10))

Number of countries: 11
Number of monthly observations: 396


,CountryName,Month,StringencyIndex_Average
0,Brazil,2020-01-01,0.538065
1,Brazil,2020-02-01,5.560000
2,Brazil,2020-03-01,43.996452
3,Brazil,2020-04-01,75.001667
4,Brazil,2020-05-01,80.541290
5,Brazil,2020-06-01,77.310000
6,Brazil,2020-07-01,79.974516
7,Brazil,2020-08-01,72.000968
8,Brazil,2020-09-01,69.168667
9,Brazil,2020-10-01,62.651290


In [50]:
fig = px.line(
    stringency_monthly,
    x="Month",
    y="StringencyIndex_Average",
    color="CountryName",
    title="Government Policy Stringency Across Countries, 2020–2022",
    labels={
        "Month": "Month",
        "StringencyIndex_Average": "Monthly Average Stringency Index (0–100)",
        "CountryName": "Country"
    }
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend_title_text="Country"
)

fig.update_yaxes(range=[0, 100])
fig.show(renderer="browser")

**Interpretation:** Government policy stringency rose sharply across most countries in early 2020, but the timing and intensity of responses differed. Many countries gradually reduced stringency during 2021–2022, while China maintained relatively high levels for longer. These differences show that governments followed varied policy trajectories during the pandemic; the index alone cannot establish which approach was most effective.

## 2. Economic Support Policies Over Time (2020–2022)

Alongside public-health restrictions, governments introduced measures to support households and businesses during the pandemic. These included income support and debt or contract relief.

The OxCGRT Economic Support Index summarizes selected government economic support measures on a scale from 0 to 100. It reflects the recorded scope of these policies, not the amount of money spent or the total financial support received by individuals and businesses.

This section compares monthly average economic support index values across the 11 countries from 2020 to 2022 to explore how government support policies changed over time.

In [51]:
# Calculate monthly average economic support

support_monthly = (
    policy.groupby(
        [
            "CountryName",
            pd.Grouper(key="Date", freq="MS")
        ],
        as_index=False
    )["EconomicSupportIndex"]
    .mean()
    .rename(columns={"Date": "Month"})
)

print("Number of countries:", support_monthly["CountryName"].nunique())
print("Number of monthly observations:", len(support_monthly))

display(support_monthly.head(10))

Number of countries: 11
Number of monthly observations: 396


,CountryName,Month,EconomicSupportIndex
0,Brazil,2020-01-01,0.000000
1,Brazil,2020-02-01,0.000000
2,Brazil,2020-03-01,12.096774
3,Brazil,2020-04-01,49.166667
4,Brazil,2020-05-01,50.000000
5,Brazil,2020-06-01,50.000000
6,Brazil,2020-07-01,50.000000
7,Brazil,2020-08-01,50.000000
8,Brazil,2020-09-01,38.333333
9,Brazil,2020-10-01,50.000000


In [52]:
fig = px.line(
    support_monthly,
    x="Month",
    y="EconomicSupportIndex",
    color="CountryName",
    title="Government Economic Support Across Countries, 2020–2022",
    labels={
        "Month": "Month",
        "EconomicSupportIndex": "Monthly Average Economic Support Index (0–100)",
        "CountryName": "Country"
    }
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend_title_text="Country"
)

fig.update_yaxes(range=[0, 100])
fig.show(renderer="browser")

**Interpretation:** The Economic Support Index shows substantial differences in the timing, reported scope, and persistence of government support measures across countries during 2020–2022. Some countries maintained relatively stable index levels for extended periods, while others showed sharp changes. These differences reflect variation in recorded support policies, but the index does not measure actual spending or the amount of financial assistance received by households and businesses. Sudden changes should therefore be interpreted cautiously.

## 3. Comparing Economic Support and Policy Stringency

Governments combined public-health restrictions with different levels of economic support during the COVID-19 pandemic. Comparing these policy measures helps us explore how government responses differed across countries.

This section compares the average Stringency Index and Economic Support Index across countries over 2020–2022. A grouped bar chart will show differences in the reported scope of restrictions and economic support measures.

The indices measure different aspects of government policy. The comparison is descriptive and does not establish whether particular combinations of policies were more effective.

In [53]:
# Compare average policy indices across countries, 2020–2022

country_policy_avg = (
    policy.groupby("CountryName", as_index=False)
    .agg(
        avg_stringency=("StringencyIndex_Average", "mean"),
        avg_economic_support=("EconomicSupportIndex", "mean")
    )
)

country_policy_long = country_policy_avg.melt(
    id_vars="CountryName",
    value_vars=["avg_stringency", "avg_economic_support"],
    var_name="Policy_Type",
    value_name="Average_Index"
)

country_policy_long["Policy_Type"] = country_policy_long["Policy_Type"].map({
    "avg_stringency": "Stringency Index",
    "avg_economic_support": "Economic Support Index"
})

display(country_policy_avg.sort_values("CountryName"))

,CountryName,avg_stringency,avg_economic_support
0,Brazil,48.542582,15.659215
1,China,73.313212,51.220347
2,Germany,44.042117,56.135949
3,India,57.865192,30.770985
4,Saudi Arabia,45.571323,31.158759
5,South Africa,43.855292,48.197993
6,South Korea,40.751040,33.234489
7,Spain,44.958969,62.157847
8,Sweden,36.689361,28.113595
9,Thailand,42.963814,54.949818


In [54]:
fig = px.bar(
    country_policy_long,
    x="CountryName",
    y="Average_Index",
    color="Policy_Type",
    barmode="group",
    title="Average Policy Stringency and Economic Support by Country, 2020–2022",
    labels={
        "CountryName": "Country",
        "Average_Index": "Average Index (0–100)",
        "Policy_Type": "Policy Measure"
    },
    color_discrete_sequence=["#3569A8", "#E69F00"]
)

fig.update_layout(
    template="plotly_white",
    xaxis_tickangle=-35,
    legend_title_text="Policy Measure"
)

fig.update_yaxes(range=[0, 100])
fig.show(renderer="browser")

## 4. The Stringency Paradox: Policy Choices and GDP Growth in 2020
In 2020, governments faced a difficult trade-off: limiting the spread of COVID-19 while protecting economic activity and people's livelihoods. Restrictions could disrupt businesses and employment, while the pandemic itself reduced travel, consumption, production, and investment.

However, countries did not rely on restrictions alone. Governments also introduced income support and debt or contract relief to help households and businesses manage the shock.

This section compares average policy stringency, the reported scope of economic support measures, and real GDP growth across the 11 countries in our sample during 2020. The aim is to investigate whether countries with similar levels of stringency experienced different economic outcomes alongside different levels of reported support.

The analysis is exploratory. The Economic Support Index measures the reported scope of support policies, not the amount of money spent or received. Differences in GDP growth may also reflect pandemic severity, economic structure, pre-existing vulnerabilities, and other factors. The results therefore cannot establish that restrictions or support policies caused a particular economic outcome.

In [55]:
# Calculate average policy measures for each country in 2020
policy_2020 = (
    policy[policy["Date"].dt.year == 2020]
    .groupby(["CountryCode", "CountryName"], as_index=False)
    .agg(
        avg_stringency=("StringencyIndex_Average", "mean"),
        avg_economic_support=("EconomicSupportIndex", "mean")
    )
)

# Select annual GDP growth for 2020
gdp_2020 = gdp.loc[
    gdp["year"] == 2020,
    ["ISO3", "country", "gdp_growth_pct"]
].copy()

# Merge policy measures with GDP growth
tradeoff_2020 = policy_2020.merge(
    gdp_2020,
    left_on="CountryCode",
    right_on="ISO3",
    how="inner",
    validate="one_to_one"
)

print("Countries included:", tradeoff_2020["CountryName"].nunique())
print("Country observations:", len(tradeoff_2020))

display(
    tradeoff_2020[
        [
            "CountryName",
            "avg_stringency",
            "avg_economic_support",
            "gdp_growth_pct"
        ]
    ].sort_values("gdp_growth_pct")
)

Countries included: 11
Country observations: 11


,CountryName,avg_stringency,avg_economic_support,gdp_growth_pct
3,Spain,56.272158,68.032787,-10.940071
10,South Africa,54.488169,47.062842,-6.168918
8,Thailand,46.118934,69.467213,-6.093063
4,India,64.766557,48.838798,-5.777725
2,Germany,51.812787,35.758197,-4.131914
6,Saudi Arabia,57.214235,46.072404,-3.804789
0,Brazil,57.505628,37.021858,-3.276759
9,United States,57.113907,47.814208,-2.081376
7,Sweden,49.005874,40.095628,-1.933797
5,South Korea,49.429563,38.729508,-0.700242


### Visualising the Policy Trade-off

The bubble chart compares average policy stringency with annual GDP growth in 2020. Each bubble represents one country, and its size represents the country's average Economic Support Index.

This allows us to explore whether countries with similar levels of stringency experienced different economic outcomes alongside different levels of reported support. Countries that appear unusual relative to the overall pattern will be candidates for further investigation.

The chart identifies patterns worth exploring rather than estimating the causal effect of economic support or restrictions.

In [56]:
fig = px.scatter(
    tradeoff_2020,
    x="avg_stringency",
    y="gdp_growth_pct",
    size="avg_economic_support",
    color="avg_economic_support",
    text="CountryName",
    hover_name="CountryName",
    hover_data={
        "avg_stringency": ":.1f",
        "avg_economic_support": ":.1f",
        "gdp_growth_pct": ":.2f"
    },
    size_max=34,
    color_continuous_scale="Teal",
    title="The Stringency Paradox: Policy Choices and GDP Growth in 2020",
    labels={
        "avg_stringency": "Average Policy Stringency (0–100)",
        "gdp_growth_pct": "Real GDP Growth (%)",
        "avg_economic_support": "Economic Support Index (0–100)"
    }
)

fig.update_traces(
    textposition="top center",
    marker=dict(
        opacity=0.8,
        line=dict(width=1, color="white")
    )
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    height=650,
    width=1150,
    margin=dict(l=80, r=50, t=100, b=80),
    coloraxis_colorbar=dict(title="Support Index")
)

fig.show(renderer="browser")

**Interpretation — Why did economic outcomes differ?**

China recorded positive GDP growth of 2.34% in 2020 despite having the highest average policy stringency in the selected sample. In contrast, Spain experienced a sharp contraction of 10.94%, despite a relatively high Economic Support Index. Thailand also recorded a contraction of 6.09% alongside a high support index. These contrasting outcomes show that neither policy stringency nor the reported scope of economic support alone explains countries' economic performance. Differences in economic structure, exposure to international tourism and trade, and the severity of pandemic disruptions may also have contributed. These are possible explanations to investigate, rather than effects established by this comparison.

## 5. Key Findings, Interpretation and Limitations

### Key Findings

- **Different policy trajectories:** Governments followed different approaches to public-health restrictions and economic support during 2020–2022. The timing, intensity, and persistence of these measures varied across the 11 selected countries.

- **Variation in policy stringency:** China recorded the highest average Stringency Index across 2020–2022, at **73.31**, while Sweden recorded the lowest, at **36.69**.

- **Variation in reported economic support:** Spain recorded the highest average Economic Support Index across 2020–2022, at **62.16**, followed by Germany at **56.14** and Thailand at **54.95**. These values represent the reported scope of support policies, not actual expenditure or the amount of assistance received.

- **The stringency paradox:** In 2020, China recorded positive real GDP growth of **2.34%** despite having the highest average policy stringency in the sample. Spain experienced the sharpest contraction, at **−10.94%**, despite a relatively high Economic Support Index. Thailand also experienced a substantial contraction of **−6.09%** alongside a high support index. Neither stringency nor the reported scope of support alone explains these contrasting outcomes.

### Overall Interpretation

The pandemic presented governments with two connected challenges: limiting the spread of COVID-19 and protecting economic activity and livelihoods. Countries responded with different combinations of public-health restrictions and economic support, but their economic outcomes varied substantially.

The comparison reveals a policy trade-off, not a simple policy formula. Stricter restrictions did not always coincide with the largest economic contractions, and higher reported economic support did not guarantee a smaller contraction. Economic structure, dependence on tourism and international trade, pandemic severity, global disruptions, and the design of support measures may all have influenced outcomes.

The monthly policy trends also show why timing matters: annual and multi-year averages cannot fully capture when restrictions were introduced, how long they lasted, or how they interacted with changing health and economic conditions.

Taken together, the findings show that understanding the economic consequences of COVID-19 requires looking beyond policy stringency alone. The combination and timing of restrictions, economic support, health conditions, and country-specific vulnerabilities matter. The analysis identifies differences between countries but cannot determine which policies were most effective or establish that a particular policy caused a particular economic outcome.

### Limitations

- **Selected sample:** The analysis covers 11 countries and should not automatically be generalized to all countries.
- **Policy indices:** The Stringency Index measures the strictness of selected policies, not compliance or effectiveness. The Economic Support Index measures the reported scope of support policies, not actual fiscal expenditure or the amount received by households and firms.
- **Different measures:** The two indices represent different dimensions of government policy and should not be treated as directly equivalent.
- **Economic influences:** GDP outcomes were affected by multiple factors, including pandemic disruption, economic structure, international tourism and trade, and global conditions. This analysis does not isolate the effects of individual policies.
- **Descriptive findings:** The comparisons show differences and associations, not causal effects.
- **Averages conceal timing:** Annual and multi-year averages can hide important changes in policy intensity and the timing of restrictions relative to infection waves and economic disruptions.
- **Further investigation needed:** More detailed country-level evidence on policy timing, actual fiscal spending, health conditions, and economic structure would be needed to explain the contrasting outcomes and evaluate policy effectiveness.

## 6. Dashboard Handoff

### Recommended Visuals

**1. Government Policy Stringency Over Time (2020–2022)**

Use the monthly line chart to show when countries tightened and relaxed restrictions. Highlight differences in the timing and persistence of policy responses. Note that the Stringency Index measures the strictness of selected policies, not their effectiveness or public compliance.

**2. Economic Support Policies Over Time (2020–2022)**

Use the monthly Economic Support Index line chart to compare the reported scope and persistence of government support measures. Clarify that the index does not represent actual government spending or the amount of assistance received by households and businesses.

**3. Comparing Policy Stringency and Economic Support**

Use the grouped bar chart of country-level average indices to compare policy approaches across 2020–2022. Explain that stringency and economic support measure different dimensions of government response and should not be treated as directly equivalent.

**4. The Stringency Paradox: Policy Choices and GDP Growth in 2020**

Use the bubble chart to compare average policy stringency with real GDP growth in 2020. Use bubble size or colour to represent the Economic Support Index, with a clear legend explaining the encoding. Keep the zero-GDP-growth reference line visible.

Highlight the contrast between China, which recorded positive GDP growth of **2.34%**, and Spain, which experienced a contraction of **−10.94%**. Thailand also experienced a substantial contraction of **−6.09%** despite a relatively high support index. The key message is that higher stringency or reported economic support did not consistently correspond to better GDP outcomes. These are descriptive comparisons, not evidence that particular policies caused the outcomes.

### Design and Interpretation Guidelines

- Use consistent country colours and clear country labels across charts wherever possible.
- Include chart titles, units, years, legends, and data-source notes.
- Make the zero-GDP-growth reference line visible in the bubble chart.
- Explain that the Economic Support Index measures the reported scope of support policies, not actual expenditure.
- Distinguish policy stringency from policy effectiveness and economic support coverage from actual financial assistance.
- Display missing data as unavailable rather than treating missing values as zero.
- Avoid causal claims about restrictions, support measures, and economic outcomes.
- Keep the central narrative focused on how governments combined public-health restrictions and economic support during the pandemic, and why countries experienced different economic outcomes.